In [4]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import CocoDetection
from transformers import DetrImageProcessor, DetrForSegmentation
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from pycocotools.coco import COCO
import torchvision.transforms as T

In [12]:
class CocoSegmentationDataset(Dataset):
    def __init__(self, img_folder, ann_file, processor):
        self.img_folder = img_folder
        self.coco = COCO(ann_file)
        self.img_ids = self.coco.getImgIds()
        self.processor = processor

    def __len__(self):
        return len(self.img_ids)

    def __getitem__(self, idx):
        img_id = self.img_ids[idx]
        ann_ids = self.coco.getAnnIds(imgIds=img_id)
        annotations = self.coco.loadAnns(ann_ids)
        image_info = self.coco.loadImgs(img_id)[0]

        image = Image.open(f"{self.img_folder}/{image_info['file_name']}").convert("RGB")

        width, height = image.size
        boxes = []
        class_labels = []
        
        for ann in annotations:
            x, y, w, h = ann['bbox']
            boxes.append([x / width, y / height, w / width, h / height])  # normalize
            class_labels.append(ann['category_id'])
        
        # convert to tensors
        labels = {
            "class_labels": torch.tensor(class_labels, dtype=torch.long),
            "boxes": torch.tensor(boxes, dtype=torch.float32)
        }
        
        encoding = self.processor(images=image, return_tensors="pt")
        encoding = {k: v.squeeze(0) for k, v in encoding.items()}
        encoding["labels"] = labels

        return encoding

def detr_collate_fn(batch):
    pixel_values = torch.stack([x["pixel_values"] for x in batch])
    labels = [x["labels"] for x in batch]  # no [0] here
    return {"pixel_values": pixel_values, "labels": labels}

from transformers import DetrImageProcessor, DetrForObjectDetection
# Initialize processor and model
processor = DetrImageProcessor.from_pretrained("facebook/detr-resnet-50")
model = DetrForObjectDetection.from_pretrained("facebook/detr-resnet-50")

# Dataset and loader
train_dataset = CocoSegmentationDataset('C:/Users/mza0288/Datset/train2017', 'C:/Users/mza0288/Datset/annotations/instances_train2017.json', processor)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=detr_collate_fn)

Some weights of the model checkpoint at facebook/detr-resnet-50 were not used when initializing DetrForObjectDetection: ['model.backbone.conv_encoder.model.layer1.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer2.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer3.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer4.0.downsample.1.num_batches_tracked']
- This IS expected if you are initializing DetrForObjectDetection from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DetrForObjectDetection from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


loading annotations into memory...
Done (t=0.04s)
creating index...
index created!


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

# Training loop
model.train()
epochs = 10

for epoch in range(epochs):
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()

        pixel_values = batch['pixel_values'].to(device)

        # move each tensor in labels to device
        labels = [
            {
                k: v.to(device) if isinstance(v, torch.Tensor) else v
                for k, v in label.items()
            } for label in batch['labels']
        ]

        outputs = model(pixel_values=pixel_values, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch [{epoch+1}/{epochs}] - Loss: {total_loss / len(train_loader):.4f}")

In [ ]:
# Save model
torch.save(model.state_dict(), "detr_segmentation.pth")

In [ ]:
def predict(image_path):
    model.eval()
    image = Image.open(image_path).convert("RGB")
    inputs = processor(images=image, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    result = processor.post_process_panoptic_segmentation(outputs, target_sizes=[image.size[::-1]])[0]

    plt.figure(figsize=(10, 10))
    plt.imshow(image)
    plt.imshow(result["segmentation"])
    plt.axis("off")
    plt.show()

# Example usage
predict('C:/Users/mza0288/Datset/val2017/tile_0_0_jpg.rf.2bc2a6231fb40041d7845e1918741a3d.jpg')